# 01 Wrangle

Loads the raw UniProt Excel file, adds derived variables, and saves a single clean dataset.

**Output:** `../output/full_dataset.csv` (also copied to `../dashboard/full_dataset.csv`)

| Column added | Source |
|---|---|
| `pLLPS_Class` | thresholds: High â‰¥ 0.7, Low < 0.4 |
| `GO_IDs` | UniProt REST (cached) |
| `Functional_Categories` | GO parent IDs expanded through GO DAG |
| `Functional Group` | first GO-based category, else 'Other' |
| `GO_Slim_Categories` | GO Slim mapping (generic) |
| `Functional Slim` | first GO Slim category, else 'Other' |
| `Location Categories` | UniProt subcell SL ontology terms |
| `Location_IDs` | UniProt SL accessions |
| `Location Primary` | first SL term, else 'Other' |
| `Is_Membrane` | SL hierarchy + TMD_count |
| `TMD_count` | transmembrane domain count (from cache) |

In [1]:
from pathlib import Path

import pandas as pd

from llps.data import (
    add_go_annotations,
    add_tmd_count,
    fetch_uniprot_go_annotations,
    fetch_uniprot_location_sl_ids,
    fetch_uniprot_tm_annotations,
    load_llps_data,
)
from llps.functional import add_functional_categories, add_go_slim_categories, is_membrane_protein
from llps.location import compartment_from_sl_ids, load_subcell_ontology, parse_sl_ids

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]:
    if (candidate / "Human Phase separation data.xlsx").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the project root")

RAW = ROOT / "Human Phase separation data.xlsx"
CACHE = ROOT / "data/uniprot_tm_cache.csv"
GO_CACHE = ROOT / "data/uniprot_go_cache.csv"
SL_CACHE = ROOT / "data/uniprot_location_sl_cache.csv"
OUT = ROOT / "output/full_dataset.csv"
APP_OUT = ROOT / "dashboard/full_dataset.csv"


In [2]:
df = load_llps_data(RAW)
print(f"{len(df):,} proteins  |  p(LLPS) {df['p(LLPS)'].min():.2f}â€“{df['p(LLPS)'].max():.2f}")
df.head(2)

âœ… Loaded 20366 proteins from /workspaces/mem_prot_llps/Human Phase separation data.xlsx
   p(LLPS) range: 0.060 - 1.000
20,366 proteins  |  p(LLPS) 0.06â€“1.00


,Entry,Entry name,Protein names,p(LLPS),n(DPR=> 25),Organism,Length,Function [CC],Subcellular location [CC],Involvement in disease,Cross-reference (PDB)
0,Q9Y6V0,PCLO_HUMAN,Protein piccolo (Aczonin),1.0,21,Homo sapiens (Human),5142.0,Scaffold protein of the presynaptic cytomatri...,"Cell junction, synapse, presynaptic active zo...",Pontocerebellar hypoplasia 3 (PCH3) [MIM:6080...,1UJD;
1,Q9Y566,SHAN1_HUMAN,SH3 and multiple ankyrin repeat domains protei...,1.0,8,Homo sapiens (Human),2161.0,Seems to be an adapter protein in the postsyn...,"Cytoplasm {ECO:0000250}. Cell junction, synap...",NaN,6CPI;


In [9]:
# GO annotations (cached)
needs_go = "GO_IDs" not in df.columns or df["GO_IDs"].isna().all()
if needs_go:
    go = fetch_uniprot_go_annotations(df["Entry"].tolist(), cache_path=str(GO_CACHE))
    df = add_go_annotations(df, go)

if "Subcellular location [CC]_x" in df.columns:
    df["Subcellular location [CC]"] = df["Subcellular location [CC]_x"].fillna(
        df["Subcellular location [CC]_y"]
    )
    df = df.drop(columns=["Subcellular location [CC]_x", "Subcellular location [CC]_y"])

df[["Entry", "GO_IDs"]].head(2)

,Entry,GO_IDs
0,Q9Y6V0,GO:0005509; GO:0005522; GO:0005544; GO:0005856...
1,Q9Y566,GO:0005829; GO:0005886; GO:0007616; GO:0008306...


In [4]:
# pLLPS classification
df["pLLPS_Class"] = pd.cut(
    df["p(LLPS)"],
    bins=[-float("inf"), 0.4, 0.7, float("inf")],
    labels=["Low", "Medium", "High"],
).astype(str)

df["pLLPS_Class"].value_counts()

pLLPS_Class
Low       10797
High       6568
Medium     3001
Name: count, dtype: int64

In [5]:
# Functional categories (GO-based)
df = add_functional_categories(df, go_col="GO_IDs")
df = add_go_slim_categories(df, go_col="GO_IDs", output_col="GO_Slim_Categories")

df["Functional_Categories"].value_counts().head(10)

/workspaces/mem_prot_llps/data/go/go-basic.obo: fmt(1.2) rel(2026-03-25) 41,853 Terms
Added functional categories to 20366 proteins
Categories found: ['Adhesion', 'Chaperone', 'Chromatin Remodeling', 'DNA Repair', 'GPCR', 'GTPase', 'Guanine Nucleotide Exchange Factor', 'Hydrolase', 'Ion Channel', 'Kinase', 'Ligase', 'Nuclear Receptor', 'Oxidoreductase', 'Phosphatase', 'Protease', 'RNA Processing', 'Receptor', 'Receptor Tyrosine Kinase', 'Structural', 'Synthetase', 'Transcription Factor', 'Transferase', 'Transporter']
Total category assignments: 17006
/workspaces/mem_prot_llps/data/go/goslim_generic.obo: fmt(1.2) rel(go/2026-03-25/subsets/goslim_generic.owl) 206 Terms


Functional Group
Other                   9124
Transferase             1430
Transcription Factor    1328
Hydrolase               1268
GPCR                     861
Transporter              789
Structural               711
Kinase                   659
Protease                 610
Oxidoreductase           596
Name: count, dtype: int64

In [10]:
# Location: fetch SL accession IDs from UniProt JSON API (cached)
sl_df = fetch_uniprot_location_sl_ids(df["Entry"].tolist(), cache_path=str(SL_CACHE))
df = df.merge(sl_df, on="Entry", how="left")

# Resolve SL accessions to human-readable names via SubcellOntology
ontology = load_subcell_ontology()
df["Location_Names"] = df["Location_SL_IDs"].apply(
    lambda v: [ontology.terms[a]["name"] for a in parse_sl_ids(v) if a in ontology.terms]
)

df[["Entry", "Location_SL_IDs", "Location_Names"]].head(5)


Location Primary
Cytoplasm                         3909
Nucleus                           3877
Other                             3116
Cell membrane                     2570
Secreted                          1684
Membrane                          1651
Endoplasmic reticulum membrane     540
Mitochondrion                      395
Mitochondrion inner membrane       254
Golgi apparatus membrane           241
Name: count, dtype: int64

In [11]:
# TMD count (reads from local cache â€” run scripts/analysis/enrich_dataset_with_tmd.py to populate)
if CACHE.exists():
    tm = fetch_uniprot_tm_annotations(df["Entry"].tolist(), cache_path=str(CACHE))
    df = add_tmd_count(df, tm)
else:
    df["TMD_count"] = 0
    print("âš  TM cache not found â€” TMD_count set to 0")

df["TMD_count"].value_counts().sort_index().head(10)

Loading TM annotation cache: /workspaces/mem_prot_llps/data/uniprot_tm_cache.csv
TMD_count added: 5218 proteins have â‰¥1 membrane domain (max=38)


TMD_count
0    15148
1     2376
2      322
3      153
4      409
5       88
6      206
7      983
8      111
9       64
Name: count, dtype: int64

In [12]:
# Membrane flag and compartment (SL hierarchy + TMD count)
df["Is_Membrane"] = df.apply(
    lambda r: is_membrane_protein(
        "", "", "",
        tmd_count=int(r.get("TMD_count") or 0),
        location_ids=parse_sl_ids(r.get("Location_SL_IDs", "")),
        ontology=ontology,
    ), axis=1
)

df["Compartment"] = df.apply(
    lambda r: compartment_from_sl_ids(
        r.get("Location_SL_IDs", ""),
        is_membrane=bool(r["Is_Membrane"]),
        ontology=ontology,
    ), axis=1
)

print(df["Is_Membrane"].sum(), "membrane proteins")
df["Compartment"].value_counts()


Membrane: 7,563  |  Non-membrane: 12,803


In [13]:
# Drop text-parsed location columns (replaced by API-sourced SL IDs)
_drop = [
    "All_Functional_Groups", "Function Categories", "Functional Group", "Functional Slim",
    "Transmembrane", "Intramembrane",
    "Subcellular location [CC]_x", "Subcellular location [CC]_y",
    "Location Categories", "Location_IDs", "Location Primary",
    *[c for c in df.columns if c.startswith("Is_") and c != "Is_Membrane"],
]
df = df.drop(columns=[c for c in _drop if c in df.columns])

# Save
OUT.parent.mkdir(exist_ok=True)
df.to_csv(OUT, index=False)
APP_OUT.parent.mkdir(exist_ok=True)
df.to_csv(APP_OUT, index=False)

print(f"Saved {len(df):,} proteins  |  {df.shape[1]} columns")
print(df.columns.tolist())


Saved 20,366 proteins  |  49 columns
  â†’ /workspaces/mem_prot_llps/output/full_dataset.csv
  â†’ /workspaces/mem_prot_llps/dashboard/full_dataset.csv  (dashboard)

Entry                                        str
Entry name                                   str
Protein names                                str
p(LLPS)                                  float64
n(DPR=> 25)                                int64
Organism                                     str
Length                                   float64
Function [CC]                                str
Involvement in disease                       str
Cross-reference (PDB)                        str
GO_IDs                                       str
pLLPS_Class                                  str
Functional_Categories                     object
Is_Adhesion                                 bool
Is_Chaperone                                bool
Is_Chromatin_Remodeling                     bool
Is_DNA_Repair                               bool
I